In [1]:
import meep as mp
import meep.mpb as mpb
import numpy as np
import math
import matplotlib.pyplot as plt

# ================= PARAMETERS =================
a = 0.5
r_hole = 0.15
Nx = 14
Ny = 14
kappa = 2/3
Rmax = max(Nx,Ny)*a

eps_slab = 12.0   # dielectric constant (example)

resolution = 32

# ================= LATTICE =================
geometry_lattice = mp.Lattice(
    size=mp.Vector3(2*Rmax, 2*Rmax)
)

geometry = []

# ================= FUNCTION TO ADD HOLE =================
def add_hole(x,y):
    geometry.append(
        mp.Cylinder(
            radius=r_hole,
            center=mp.Vector3(x,y),
            material=mp.air
        )
    )

# ================= SHRUNK LATTICE =================
for i in range(-Nx//2, Nx//2):
    for j in range(-Ny//2, Ny//2):

        x = i*a
        y = j*a
        r = math.sqrt(x*x + y*y)

        if r > Rmax:
            continue

        theta = math.atan2(y, x)
        if theta < 0:
            theta += 2*math.pi

        theta_deg = math.degrees(theta)

        if theta_deg <= 180:
            theta_new_deg = 60 + kappa*theta_deg
        else:
            theta_new_deg = 180 + kappa*(theta_deg - 180)

        theta_new = math.radians(theta_new_deg)

        x_new = r * math.cos(theta_new)
        y_new = r * math.sin(theta_new)

        add_hole(x_new, y_new)

# ================= MODE SOLVER =================
ms = mpb.ModeSolver(
    geometry_lattice=geometry_lattice,
    geometry=geometry,
    resolution=resolution,
    default_material=mp.Medium(epsilon=eps_slab),
    num_bands=6,
    k_points=[mp.Vector3(0,0)]  # Gamma point
)

ms.run_te()

# ================= GET Hz FIELD =================
hz = ms.get_efield(1)   # band index

plt.imshow(np.real(hz[:,:,2]).T, cmap='RdBu')
plt.colorbar()
plt.title("Re(Hz)")
plt.show()

Initializing eigensolver data
Computing 6 bands with 1e-07 tolerance
Working in 2 dimensions.
Grid size is 448 x 448 x 1.
Solving for 6 bands at a time.
Creating Maxwell data...
Mesh size is 3.
Lattice vectors:
     (14, 0, 0)
     (0, 14, 0)
     (0, 0, 1)
Cell volume = 196
Reciprocal lattice vectors (/ 2 pi):
     (0.0714286, -0, 0)
     (-0, 0.0714286, -0)
     (0, -0, 1)
Geometric objects:
     cylinder, center = (-4.28661,-2.47487,0)
          radius 0.15, height 1e+20, axis (0, 0, 1)
     cylinder, center = (-4.10487,-2.09763,0)
          radius 0.15, height 1e+20, axis (0, 0, 1)
     cylinder, center = (-3.93866,-1.72828,0)
          radius 0.15, height 1e+20, axis (0, 0, 1)
     cylinder, center = (-3.7921,-1.36748,0)
          radius 0.15, height 1e+20, axis (0, 0, 1)
     cylinder, center = (-3.67,-1.01542,0)
          radius 0.15, height 1e+20, axis (0, 0, 1)
     cylinder, center = (-3.57758,-0.671483,0)
          radius 0.15, height 1e+20, axis (0, 0, 1)
     cylinder, cen

IndexError: index 2 is out of bounds for axis 2 with size 1